# ETL_01 — Test de Conexión y Verificación de Schema
**Proyecto:** INGEI AFOLU – DNCC / MADES – Paraguay  
**Propósito:** Verificar que Python puede conectarse a `ingei_afolu` en PostgreSQL 16  
y que el schema está completo antes de comenzar la carga de datos.

---

## 0. Instalación de dependencias
Ejecutar solo la primera vez. Comentar luego.

In [ ]:
# %pip install psycopg2-binary sqlalchemy pandas python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------------------- -------------------- 1.3/2.8 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 8.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------  2.1/2.1 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 6.7 MB/s  0:00:00
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ---------- ----------------------------- 1/4 [psycopg2-binary]
   -------------------- ------------------- 2/4 [greenlet]
   -------------------- ------------------- 2/4 [greenlet]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   -----

## 1. Configuración de conexión
Editá los parámetros según tu entorno local.

In [2]:
# ── Parámetros de conexión ──────────────────────────────────
DB_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'dbname':   'ingei_afolu',
    'user':     'postgres',
    'password': 'postgres'
}

# URL para SQLAlchemy
DB_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
)

print(f"Conectando a: {DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}")

Conectando a: localhost:5432/ingei_afolu


## 2. Test de conexión con psycopg2

In [3]:
import psycopg2
import time

t0 = time.time()
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()
    cur.execute('SELECT version();')
    version = cur.fetchone()[0]
    ms = round((time.time() - t0) * 1000, 1)
    print(f'✅ Conexión exitosa ({ms} ms)')
    print(f'   {version}')
    cur.close()
    conn.close()
except Exception as e:
    print(f'❌ Error de conexión: {e}')

✅ Conexión exitosa (68.9 ms)
   PostgreSQL 16.0, compiled by Visual C++ build 1935, 64-bit


## 3. Test de conexión con SQLAlchemy
SQLAlchemy es lo que usaremos en todos los ETLs siguientes con `pandas.read_sql` y `to_sql`.

In [4]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(DB_URL)

try:
    with engine.connect() as con:
        result = con.execute(text('SELECT current_database(), current_user, now()'))
        row = result.fetchone()
    print(f'✅ SQLAlchemy OK')
    print(f'   Base de datos : {row[0]}')
    print(f'   Usuario       : {row[1]}')
    print(f'   Hora servidor : {row[2]}')
except Exception as e:
    print(f'❌ Error SQLAlchemy: {e}')

✅ SQLAlchemy OK
   Base de datos : ingei_afolu
   Usuario       : postgres
   Hora servidor : 2026-06-30 19:38:07.900176-04:00


## 4. Verificación de schemas

In [5]:
SQL_SCHEMAS = """
SELECT schema_name
FROM information_schema.schemata
WHERE schema_name IN ('staging','inventario','etf_output','auditoria')
ORDER BY schema_name
"""

df_schemas = pd.read_sql(SQL_SCHEMAS, engine)
esperados  = {'staging', 'inventario', 'etf_output', 'auditoria'}
encontrados = set(df_schemas['schema_name'])

if esperados == encontrados:
    print(f'✅ Los 4 schemas están presentes: {sorted(encontrados)}')
else:
    faltantes = esperados - encontrados
    print(f'❌ Schemas faltantes: {faltantes}')

✅ Los 4 schemas están presentes: ['auditoria', 'etf_output', 'inventario', 'staging']


## 5. Verificación de tablas

In [6]:
SQL_TABLAS = """
SELECT table_schema AS schema,
       table_name   AS tabla,
       table_type   AS tipo
FROM information_schema.tables
WHERE table_schema IN ('staging','inventario','etf_output','auditoria')
ORDER BY table_schema, table_name
"""

df_tablas = pd.read_sql(SQL_TABLAS, engine)

TABLAS_ESPERADAS = {
    'staging.raw_afolu_datos',
    'staging.raw_clima_datos',
    'staging.raw_sector_json',
    'inventario.dim_versiones',
    'inventario.dim_anhos',
    'inventario.dim_departamentos',
    'inventario.dim_gases',
    'inventario.dim_categorias_ipcc',
    'inventario.dim_variables_etf',
    'inventario.dim_tipos_dato',
    'inventario.fact_datos_actividad',
    'inventario.fact_valores_etf',
    'inventario.ref_nk_codigos',
    'etf_output.json_generados',
    'auditoria.log_pipeline',
    'auditoria.control_calidad',
}

encontradas = set(df_tablas['schema'] + '.' + df_tablas['tabla'])
faltantes   = TABLAS_ESPERADAS - encontradas

print(f'Objetos encontrados: {len(df_tablas)}')
if not faltantes:
    print(f'✅ Las 16 tablas están presentes')
else:
    print(f'❌ Tablas faltantes: {faltantes}')

df_tablas

Objetos encontrados: 19
✅ Las 16 tablas están presentes


,schema,tabla,tipo
0,auditoria,control_calidad,BASE TABLE
1,auditoria,log_pipeline,BASE TABLE
2,etf_output,json_generados,BASE TABLE
3,etf_output,v_cobertura_inventario,VIEW
4,inventario,dim_anhos,BASE TABLE
5,inventario,dim_categorias_ipcc,BASE TABLE
6,inventario,dim_departamentos,BASE TABLE
7,inventario,dim_gases,BASE TABLE
8,inventario,dim_tipos_dato,BASE TABLE
9,inventario,dim_variables_etf,BASE TABLE


## 6. Verificación de datos iniciales

In [7]:
checks = {
    'dim_versiones'    : ('SELECT COUNT(*) FROM inventario.dim_versiones',      4),
    'dim_anhos'        : ('SELECT COUNT(*) FROM inventario.dim_anhos',          35),
    'dim_departamentos': ('SELECT COUNT(*) FROM inventario.dim_departamentos',  17),
    'dim_gases'        : ('SELECT COUNT(*) FROM inventario.dim_gases',           4),
    'ref_nk_codigos'   : ('SELECT COUNT(*) FROM inventario.ref_nk_codigos',      7),
    'version_activa'   : (
        "SELECT COUNT(*) FROM inventario.dim_versiones WHERE activa = TRUE", 1
    ),
}

all_ok = True
with engine.connect() as con:
    for nombre, (sql, esperado) in checks.items():
        resultado = con.execute(text(sql)).scalar()
        ok = resultado == esperado
        estado = '✅' if ok else '❌'
        print(f'{estado} {nombre:<20} → encontrado: {resultado:>3}  esperado: {esperado}')
        if not ok:
            all_ok = False

print()
if all_ok:
    print('✅ Todos los datos iniciales están correctos. Listo para ETL_02.')
else:
    print('❌ Hay discrepancias. Revisar antes de continuar.')

✅ dim_versiones        → encontrado:   4  esperado: 4
✅ dim_anhos            → encontrado:  35  esperado: 35
✅ dim_departamentos    → encontrado:  17  esperado: 17
✅ dim_gases            → encontrado:   4  esperado: 4
✅ ref_nk_codigos       → encontrado:   7  esperado: 7
✅ version_activa       → encontrado:   1  esperado: 1

✅ Todos los datos iniciales están correctos. Listo para ETL_02.


## 7. Verificación de versión activa

In [8]:
SQL_VERSION = """
SELECT version_id, nombre_version, anho_base_desde,
       anho_base_hasta, activa, descripcion
FROM inventario.dim_versiones
ORDER BY version_id
"""
df_ver = pd.read_sql(SQL_VERSION, engine)
df_ver

,version_id,nombre_version,anho_base_desde,anho_base_hasta,activa,descripcion
0,1,1BTR,1990,2021,False,Primer Informe Bienal de Transparencia – línea...
1,2,5NC,1990,2021,False,5ta Comunicación Nacional – misma serie que 1BTR
2,3,2BTR,1990,2022,False,Segundo Informe Bienal de Transparencia
3,4,3BTR,1990,2024,True,3er Informe Bienal de Transparencia – versión ...


## 8. Test de escritura (insert + rollback)
Verifica que el usuario tiene permisos de escritura. Hace un INSERT y lo revierte.

In [9]:
SQL_INSERT_TEST = text("""
INSERT INTO auditoria.log_pipeline
    (etapa, resultado, mensaje, detalle)
VALUES
    ('conexion_test', 'ok', 'ETL_01: test de escritura exitoso',
     '{"notebook": "ETL_01_conexion_test"}'::jsonb)
RETURNING log_id
""")

try:
    with engine.begin() as con:
        row = con.execute(SQL_INSERT_TEST).fetchone()
        log_id = row[0]
    print(f'✅ Escritura OK — log_id generado: {log_id}')
    print('   (registro guardado en auditoria.log_pipeline)')
except Exception as e:
    print(f'❌ Error de escritura: {e}')

✅ Escritura OK — log_id generado: 1
   (registro guardado en auditoria.log_pipeline)


## 9. Resumen final

In [10]:
print('=' * 55)
print('  RESUMEN ETL_01 — INGEI AFOLU')
print('=' * 55)
print(f'  Host       : {DB_CONFIG["host"]}:{DB_CONFIG["port"]}')
print(f'  Base datos : {DB_CONFIG["dbname"]}')
print(f'  Usuario    : {DB_CONFIG["user"]}')
print()
print('  Siguiente paso → ETL_02_poblar_dim_variables_etf.ipynb')
print('  Pobla dim_variables_etf desde metadata.json.lzma')
print('  del etf-cli (32.694 variables AFOLU)')
print('=' * 55)

engine.dispose()
print('\n  Conexión cerrada.')

  RESUMEN ETL_01 — INGEI AFOLU
  Host       : localhost:5432
  Base datos : ingei_afolu
  Usuario    : postgres

  Siguiente paso → ETL_02_poblar_dim_variables_etf.ipynb
  Pobla dim_variables_etf desde metadata.json.lzma
  del etf-cli (32.694 variables AFOLU)

  Conexión cerrada.
